In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## CARGA DE DATOS Y CONFIGURACIÓN INICIAL Y VISUAL

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,5)

ruta = "../data/raw/Calidad_Del_Aire_En_Colombia_(Promedio_Anual)_20260318.csv"

df = pd.read_csv(ruta)

df.head()

In [ ]:
df.info()

df.describe()

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

## REVISIÓN DE VALORES FALTANTES

In [ ]:
nulos = df.isnull().sum()

porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

tabla_nulos = pd.DataFrame({
    "Nulos": nulos,
    "Porcentaje (%)": porcentaje_nulos
})

tabla_nulos[tabla_nulos["Nulos"] > 0]

## Conclusiones
- La gran mayoría de las variables no presentan valores faltantes,  esto nos permite indentificar que el dataset es de calidad y presenta una buena completitud en la data. 
- Según el analisis solo 3 variables presentaron datos faltantes, las cuales son: 
  * Representatividad Temporal : 148 valores faltantes (~0.50%)
  * Código del Municipio : 4 valores faltantes (~0.01%)
  * Tipo de Estación : 2 valores faltantes (~0.006%)

Como los datos faltantes del dataset son muy bajos inferiores al 1% no representa un problema para este analisis. Aunque tenemos una variable con un 0,5% de valores faltantes (Representatividad Temporal) no tiende a afectar en gran medida  nuestro analisis debido a su proporcion respecto a los datos totales. En general el restante de las variables tienen una proporción de faltantes depresiables. 

# Estrategia de tratamiento

Se recomienda eliminar los registros con valores faltantes, debido a que no superamn el 1% del total de los datos por lo cual no se tendria un impacto signioficativo como sesgo en el resultado final. 

## Conclusión final 
Debido a que el dataset tiene una excelente calidad se procede con el analisis exploratorio y modelado sin necesidad de un tratamiento intensivo en los valores faltantes. 

## VISUALIZACIÓN FALTANTES

In [ ]:
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Mapa de valores faltantes")
plt.show()

In [ ]:
df_clean = df.dropna()

print("Antes:", df.shape)
print("Después:", df_clean.shape)

In [ ]:

excluir = [
    "Latitud",
    "Longitud",
    "Código del Departamento",
    "Código del Municipio"
]

columnas_numericas = df_clean.select_dtypes(include=[np.number]).columns

columnas_analisis = [col for col in columnas_numericas if col not in excluir]

## IMPLEMENTAMOS EL MÉTODO IQR



Se opta por el uso del método IQR, ya que no asume una distribución normal de los datos y es más robusto frente a valores extremos.

Dado que las variables de calidad del aire suelen presentar distribuciones sesgadas debido a factores ambientales, el uso de Z-score podría generar la eliminación de valores que representan eventos reales.

Por esta razón, el método IQR es más adecuado para este análisis.

Adicionalmente, se considera que algunos valores atípicos pueden corresponder a episodios reales de contaminación, por lo que no se eliminan automáticamente, sino que se analizan antes de tomar una decisión.

In [ ]:

columnas_numericas = df_clean.select_dtypes(include=[np.number]).columns

df_iqr = df_clean.copy()

outliers_iqr = pd.DataFrame()

for col in columnas_numericas:
    Q1 = df_iqr[col].quantile(0.25)
    Q3 = df_iqr[col].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    condicion = (df_iqr[col] < limite_inferior) | (df_iqr[col] > limite_superior)

    outliers_iqr[col] = condicion


outliers_iqr.sum()

## TRATAMIENTO DE OUTLIERS POR VARIABLE

Al implementar el metodo IQR  se identificó un número considerable de outliers en las variables que se relacionan con las excedencias de contaminación. Sin embargo se decidio que: 

- Las variables de ubicación geografica no presentan un riesgo alto debido a su naturalidad ya que estas representan ubicación y no valores especificos. 
- Los valores asignados en variables de la calidad de aire, pueden corresponder a eventos reales de contaminación más no a errores. 

## DECISIÓN

No se realizara una eliminación automatica de los outliers, estos valores pueden tener información relevante para nuestro analisis de contaminación ambiental. En lugar de realizar un tratamiento como eliminación o modificación de estos datos realizamos un analisis visual por medio de Boxplot para entender su comportamiento. 

Debido a que el dataset tiene muchos valores atipicos eliminar estos pondria en riesgo la calidad de nuestro dataset, eliminando datos importantes que podremos usar en nuestro analisis (insights). 

## ANALISIS DE OUTLIERS CON GRÁFICAS BOXPLOT

In [ ]:
for col in columnas_analisis:
    plt.figure()
    sns.boxplot(x=df_clean[col])
    plt.title(f"Boxplot - {col}")
    plt.show()

## Analisis de graficas boxplot

El análisis gráfico mediante boxplots permitió identificar diferentes comportamientos en las variables del dataset.

- Variables como el tiempo de exposición y la representatividad temporal presentan distribuciones estables y sin valores atípicos relevantes.
- Por otro lado, las variables relacionadas con excedencias de contaminación muestran distribuciones altamente sesgadas a la derecha.

Esto indica que, aunque la mayoría de las observaciones presentan niveles bajos de contaminación, existen casos extremos que representan eventos críticos.

Estos valores atípicos no deben considerarse errores, sino información relevante para el análisis, ya que reflejan condiciones reales del entorno ambiental.

En consecuencia, no se eliminaron los outliers, sino que se mantuvieron para preservar la integridad del análisis.

In [ ]:
import os

df_final = df_clean.copy()

ruta = os.path.join("..", "data", "processed", "df_limpio.csv")
os.makedirs(os.path.dirname(ruta), exist_ok=True)

df_final.to_csv(ruta, index=False)

print("guardado correctamente")